# Negation

Adapted from Qualtran v0.7.0 commit bcb17bb25360ed68ce3bc084645db3e929799389. Copyright 2024 Google LLC, Apache-2.0.

In [ ]:
from IPython.display import SVG, display
from examples.algorithms.negation_catalog import negation_example_graph, negation_examples
from examples.algorithms.field_addition_catalog import field_signature_svg
from flagquantum.drawer.call_graph import export_call_graph_dot, render_call_graph_svg

## `Negate`
Compute the two's complement negation for a integer/fixed-point value.

This bloq is equivalent to the "Unary minus" [1] C++ operator.
- For a signed `x`, the output is `-x`.
- For an unsigned `x`, the output is `2^n - x` (where `n` is the bitsize).

This is computed by the bit-fiddling trick `-x = ~x + 1`, as follows:
1. Flip all the bits (i.e. `x := ~x`)
2. Add 1 to the value (interpreted as an unsigned integer), ignoring
   any overflow. (i.e. `x := x + 1`)

For a controlled negate bloq: the second step uses a quantum-quantum adder by
loading the constant (i.e. 1), therefore has an improved controlled version
which only controls the constant load and not the adder circuit, hence halving
the T-cost compared to a controlled adder.

#### Parameters
 - `dtype`: The data type of the input value. 

#### Registers
 - `x`: Any unsigned value or signed value (in two's complement form). 

#### References
 - [Arithmetic Operators - cppreference](https://en.cppreference.com/w/cpp/language/operator_arithmetic). Operator "Unary Minus". Last accessed 17 July 2024.


Unsigned output is reduced modulo the word size, so zero remains zero. Words are big-endian and the original factory types are integer words. The symbolic width describes the signature and graph. Concrete lowering owns the explicit clean constant and carry wires.

In [ ]:
examples = negation_examples()

### Example Instances

In [ ]:
concrete = examples[0]
concrete

In [ ]:
symbolic = examples[1]
symbolic

#### Graphical Signature

In [ ]:
display(SVG(field_signature_svg([(name, signature, {}) for name, signature, _ in examples])))

### Call Graph

In [ ]:
graph = negation_example_graph(concrete)
titles = {"bitwise_not": "BitwiseNot(8)", "constant_add": "AddK(8, 1)"}
display(SVG(render_call_graph_svg(export_call_graph_dot(
    graph.root, graph.direct_calls,
    title=lambda node: titles[node.operation] if node != graph.root else str(node),
))))
print("Source-declared leaves", graph.leaf_counts)

## Coherent negative control through append

A zero control enables negation and a one control bypasses it. The carry network runs in both branches. The signed two-bit minimum and zero are fixed points of the two's-complement permutation. A second application restores the state. The complex trajectory assertions include all work wires. The primitive counts and independent AddK-based source model are printed separately.

In [ ]:
from collections import Counter
import torch
from examples.algorithms.binary_field_arithmetic import FieldOperation
from flagquantum.compiler.registers.arithmetic.controlled_addition import controlled_negate_ir
from flagquantum.compiler.registers.arithmetic.constant_add_resource_model import (
    source_controlled_constant_add_resource_formula,
)
from flagquantum.dynamic import DynamicCircuit
from flagquantum.runtime.dynamic.execution import run_dynamic

gates = controlled_negate_ir(0, (1, 2), (3, 4), (5,), (0,), control_value=0)
state = torch.zeros(64, dtype=torch.complex128)
state[1 << 3] = 0.8
state[5 << 3] = 0.6j
expected = state.clone()
expected[3 << 3], expected[1 << 3] = state[1 << 3], 0
circuit = DynamicCircuit(6, inputs=state, dtype=torch.complex128)
op = FieldOperation(6, gates)
circuit.append(op)
computed = run_dynamic(circuit, shots=8, seed=29, strategy="trajectory")
torch.testing.assert_close(computed.final_states.reshape(8, -1), expected.expand(8, -1), atol=1e-12, rtol=0)
circuit.append(op)
restored = run_dynamic(circuit, shots=8, seed=29, strategy="trajectory")
torch.testing.assert_close(restored.final_states.reshape(8, -1), state.expand(8, -1), atol=1e-12, rtol=0)
print("Emitted primitives", dict(Counter(gate.name for gate in gates)))
model = source_controlled_constant_add_resource_formula(2, 1, control_value=0)
print("Controlled increment model and two complement CX", model)